In [0]:
%pip install requests

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA notebook_breweries;

In [0]:
%skip
import requests
url = "https://api.openbrewerydb.org/v1/breweries/meta"
meta = requests.get(url).json()

display(meta)

In [0]:
import requests

def fetch_all_breweries():
    all_breweries = []
    page = 1
    per_page = 200

    while True:
        url = "https://api.openbrewerydb.org/v1/breweries"
        params = { "per_page" : per_page, "page" : page}

        response = requests.get(url, params = params)
        data = response.json()

        # Se la pagina è vuota, abbiamo finito
        if not data: 
            break

        all_breweries.extend(data)  #aggiungi i record alla lista
        #.extend() aggiunge tutti gli elementi di una lista in coda a un'altra lista  
        #è diverso da .append() che aggiunge l'intera lista come un singolo elemento
        
        # Se la pagina ha meno di 200 record, è l'ultima
        if len(data) < per_page:
            break
        
        page += 1
        
    return all_breweries

df_raw = spark.createDataFrame(fetch_all_breweries())

In [0]:
import requests
import json
from pyspark.sql import SparkSession

url = "https://api.openbrewerydb.org/v1/breweries?per_page=200"

response = requests.get(url)
data = response.json()

spark = SparkSession.builder.appName("Breweries").getOrCreate()
df_raw = spark.createDataFrame(data)

In [0]:
# Save as a managed Delta table in Unity Catalog (replace with your catalog/schema if needed)
catalog = "workspace"
schema = "notebook_breweries"
table_name = "bronze_breweries"

df_raw.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.{table_name}")